# Wikidata Definition Filter — Demo

Fetch raw Wikidata candidates for a term, then use an LLM (local server **or** OpenAI API) to:
1. Drop candidates whose meaning is not actually the term itself.
2. Merge candidates whose meanings substantially overlap.
3. Reorder the survivors by everyday-usage frequency.

The cache (`cache/wikidata_definition_filter_cache.sqlite3`) is keyed **only** by the normalized term — once a term has been processed, the cached answer is reused regardless of model/language/num_candidates. Pass `use_cache=False` to force a fresh run that overwrites the cached entry.

In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print('cwd =', PROJECT_ROOT)

cwd = /home/xiaoyue/LiteSemRAG


In [2]:
import time
import pandas as pd
from wikidata_definition_filter import (
    WikidataDefinitionFilter,
    fetch_wikidata_candidates,
)

# Pick the backend: set USE_API=True to use OpenAI (gpt-4o-mini, key from API_KEY file).
# Set USE_API=False to use the local OpenAI-compatible server.
USE_API = True

wd_filter = WikidataDefinitionFilter(use_api=USE_API)
print('provider:', wd_filter.provider)
print('model   :', wd_filter.model)
print('cache   :', wd_filter.cache_path)

provider: openai
model   : gpt-4o-mini
cache   : cache/wikidata_definition_filter_cache.sqlite3


## 1. Inspect the raw Wikidata candidates (no filtering)

These are the senses returned directly by Wikidata's `wbsearchentities`, with `detailed_description` from Wikipedia attached. Note how some entries are clearly unrelated (songs, films, people whose name happens to contain the term) and others overlap heavily.

In [3]:
TERM = 'bank'
NUM_CANDIDATES = 10

raw_candidates = fetch_wikidata_candidates(TERM, num_candidates=NUM_CANDIDATES)
raw_df = pd.DataFrame([
    {
        'idx': c.index,
        'id': c.entity_id,
        'label': c.label,
        'description': c.description,
        'detailed_description': (c.detailed_description[:200] + ' ...') if len(c.detailed_description) > 200 else c.detailed_description,
    }
    for c in raw_candidates
])
raw_df

,idx,id,label,description,detailed_description
0,0,Q22687,bank,financial institution that accepts deposits,A bank is a financial institution that accepts...
1,1,Q16479798,Bank,family name,Bank is a surname.
2,2,Q468756,shore,fringe of land at the edge of a large body of ...,"A coast (also called the coastline, shoreline,..."
3,3,Q104869157,Bank DLR station,Docklands Light Railway station,
4,4,Q1377943,ocean bank,part of the sea which is shallow compared to i...,"An ocean bank, sometimes referred to as a fish..."
5,5,Q13499,debit card,payment card connected directly to a cardholde...,"A debit card, also known as a check card, cheq..."
6,6,Q806825,Banks,family name,Banks is an English surname.
7,7,Q2897058,bank,"in geography, area between high and low tide m...","In geography, a bank is the land alongside a b..."
8,8,Q131258,Banksia,genus of plants,Banksia is a genus of around 170 species of fl...
9,9,Q806704,Bank and Monument stations,London Underground and Docklands Light Railway...,Bank and Monument are two interlinked stations...


## 2. Run the LLM filter (cache miss, then cache hit)

In [4]:
t0 = time.time()
result_fresh = wd_filter.filter_definitions(TERM, num_candidates=NUM_CANDIDATES, use_cache=False)
t_fresh = time.time() - t0

t0 = time.time()
result_cached = wd_filter.filter_definitions(TERM, num_candidates=NUM_CANDIDATES, use_cache=True)
t_cached = time.time() - t0

print(f'fresh run  : {t_fresh:5.2f}s   from_cache={result_fresh.from_cache}')
print(f'cached run : {t_cached:5.2f}s   from_cache={result_cached.from_cache}')

fresh run  :  4.52s   from_cache=False
cached run :  0.00s   from_cache=True


In [5]:
def show_result(result):
    rows = []
    for rank, defn in enumerate(result.definitions, start=1):
        rows.append({
            'rank': rank,
            'definition': defn.definition,
            'sources': ' / '.join(f'{lab} ({eid})' for lab, eid in zip(defn.source_labels, defn.source_entity_ids)),
            'merged': defn.is_merged,
            'rewritten': defn.is_rewritten,
        })
    return pd.DataFrame(rows)

show_result(result_fresh)

,rank,definition,sources,merged,rewritten
0,1,financial institution that accepts deposits,bank (Q22687),False,False
1,2,"in geography, area between high and low tide m...",bank (Q2897058),False,False


## 3. Compare a few polysemous terms

Each term is processed once and then cached. Re-running the cell is essentially free.

In [6]:
for term in ['apple', 'python', 'mercury', 'crane']:
    print('=' * 80)
    print(f'TERM: {term}')
    res = wd_filter.filter_definitions(term, num_candidates=10, use_cache=False)
    print(f'  raw candidates : {len(res.candidates)}')
    print(f'  kept senses    : {len(res.definitions)}   (from_cache={res.from_cache})')
    for i, d in enumerate(res.definitions, start=1):
        marker = '[merged]' if d.is_merged else '        '
        print(f'  {i}. {marker} {d.definition}')
        print(f'              sources: {d.source_labels}')

TERM: apple
  raw candidates : 10
  kept senses    : 2   (from_cache=False)
  1. [merged] An apple is the round, edible fruit of an apple tree (Malus spp.), cultivated worldwide.
              sources: ['apple', 'Malus pumila']
  2.          American multinational technology company based in Cupertino, California
              sources: ['Apple Inc.']
TERM: python
  raw candidates : 10
  kept senses    : 2   (from_cache=False)
  1.          general-purpose programming language
              sources: ['Python']
  2.          genus of reptiles
              sources: ['Python']
TERM: mercury
  raw candidates : 10
  kept senses    : 4   (from_cache=False)
  1.          chemical element with symbol Hg and atomic number 80
              sources: ['mercury']
  2.          first planet from the Solar System and smallest among all, tellurian and with extreme temperatures
              sources: ['Mercury']
  3.          Roman god of trade, merchants, thieves and travel
              sources: ['Me

In [7]:
term = 'director'
print('=' * 80)
print(f'TERM: {term}')
res = wd_filter.filter_definitions(term, num_candidates=10, use_cache=False)
print(f'  raw candidates : {len(res.candidates)}')
print(f'  kept senses    : {len(res.definitions)}   (from_cache={res.from_cache})')
for i, d in enumerate(res.definitions, start=1):
    marker = '[merged]' if d.is_merged else '        '
    print(f'  {i}. {marker} {d.definition}')
    print(f'              sources: {d.source_labels}')

TERM: director
  raw candidates : 10
  kept senses    : 2   (from_cache=False)
  1. [merged] a person who directs the artistic or dramatic aspects of a performance or production
              sources: ['film director', 'theatrical director']
  2.          person who leads a particular area of a company or organization
              sources: ['director']


## 4. Force-refresh a cached entry

Calling with `use_cache=False` re-runs the LLM and overwrites the cached payload. Subsequent calls with `use_cache=True` will read the refreshed entry, regardless of which model produced it.

In [8]:
before = wd_filter.filter_definitions('apple', num_candidates=10, use_cache=True)
after  = wd_filter.filter_definitions('apple', num_candidates=10, use_cache=False)
print('before from_cache:', before.from_cache)
print('after  from_cache:', after.from_cache, '(forced refresh; cache now overwritten)')

again = wd_filter.filter_definitions('apple', num_candidates=10, use_cache=True)
print('again  from_cache:', again.from_cache, '(should be True, served from refreshed cache)')

before from_cache: True
after  from_cache: False (forced refresh; cache now overwritten)
again  from_cache: True (should be True, served from refreshed cache)


## 5. Switch to the other backend

You can construct a second filter pointing at the local server (or the API) without losing the cache, since the cache key is just the term.

In [9]:
alt_filter = WikidataDefinitionFilter(use_api=not USE_API)
print('alt provider:', alt_filter.provider)
print('alt model   :', alt_filter.model)

# Cached entry from the first backend is reused here.
shared = alt_filter.filter_definitions('apple', num_candidates=10, use_cache=True)
print('from_cache  :', shared.from_cache)
print('cached metadata:', shared.metadata)

alt provider: local
alt model   : meta-llama/Meta-Llama-3-8B-Instruct
from_cache  : True
cached metadata: {'provider': 'openai', 'model': 'gpt-4o-mini'}
